In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

import os
import json

results = []
if os.path.exists("../reports/results.json"):
    try:
        with open("../reports/results.json") as f:
            results = json.load(f)
    except (json.JSONDecodeError, ValueError):
        results = []
df = pd.read_parquet('../data/loans_clean.parquet')

def log_run(results, entry):
    results = [r for r in results if r["run_name"] != entry["run_name"]]
    results.append(entry)
    return results

df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df["credit_history_months"] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df["annual_inc"].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

df_train = df[df['issue_year'].isin([2014, 2015])].copy()
df_val   = df[df["issue_year"] == 2016].copy()
df_test  = df[df['issue_year'] == 2017].copy()

y_train = df_train['default'].values
y_val   = df_val['default'].values

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

Train: 598,647  |  Val: 293,095  |  Test: 169,300


In [2]:
gain_rate = 0.10
loss_rate = 0.50

def calculate_profit(df_subset, approval_mask):
    """Total profit from a set of approval decisions."""
    approved = df_subset[approval_mask]
    if len(approved) == 0:
        return 0
    profit = np.where(
        approved["default"] == 0,
         gain_rate * approved["loan_amnt"],
        -loss_rate * approved["loan_amnt"]
    )
    return profit.sum()

In [3]:
base_rate = float(df_train["default"].mean())
probs = np.full(len(df_val), base_rate)
profit = calculate_profit(df_val, np.ones(len(df_val), dtype=bool))

results = log_run(results, {
    "run_name": "baseline_approve_all",
    "model_type": "constant",
    "predicted_default_prob": base_rate,
    "auc": 0.5,
    "pr_auc": float(y_val.mean()),
    "brier": float(brier_score_loss(y_val, probs)),
    "profit_at_threshold": float(profit),
    "approval_rate": 1.0,
})

print("Logged baseline_approve_all")

Logged baseline_approve_all


In [4]:
grade_rates = df_train.groupby("grade")["default"].mean()
val_probs = df_val["grade"].map(grade_rates).values

thresholds = np.linspace(0.05, 0.55, 100)
profits = [calculate_profit(df_val, val_probs < t) for t in thresholds]
best_idx = int(np.argmax(profits))
best_threshold = float(thresholds[best_idx])
best_profit = float(profits[best_idx])
approval_rate = float((val_probs < best_threshold).mean())

profits = np.array(profits)
best_idx = profits.argmax()
results = log_run(results, {
    "run_name": "baseline_grade_only",
    "model_type": "grade_lookup",
    "best_threshold": best_threshold,
    "auc": float(roc_auc_score(y_val, val_probs)),
    "pr_auc": float(average_precision_score(y_val, val_probs)),
    "brier": float(brier_score_loss(y_val, val_probs)),
    "profit_at_threshold": best_profit,
    "approval_rate": approval_rate,
})

print(f"Logged baseline_grade_only — AUC: {results[-1]['auc']:.4f}, Profit: ${best_profit:,.0f}")

Logged baseline_grade_only — AUC: 0.6816, Profit: $42,848,472


In [5]:
df_train_fe = df_train.copy()
df_val_fe = df_val.copy()
df_test_fe = df_test.copy()

def prep_features(df):
    """Apply feature engineering decisions from EDA."""
    df = df.copy()
    
    redundant = [
        "fico_range_high",
        "funded_amnt",
        "funded_amnt_inv",
        "num_sats",
        "installment",
        "num_rev_tl_bal_gt_0",
    ]

    joint_cols = [c for c in df.columns if c.startswith("sec_app") or c.endswith("_joint")]

    high_cardinality = ["zip_code", "sub_grade"]

    split_cols = ["issue_year"]

    emp_map = {
        "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
        "5 years": 5, "6 years": 6, '7 years': 7, "8 years": 8, "9 years": 9,
        "10+ years": 10
    }
    df["emp_length"] = df["emp_length"].map(emp_map)
    
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    
    df = df.drop(
        columns=[c for c in cols_to_drop if c in df.columns]
    )
    
    return df



df_train_fe = prep_features(df_train_fe)
df_val_fe = prep_features(df_val_fe)
df_test_fe = prep_features(df_test_fe)

print(f"Features after prep: {df_train_fe.shape[1]}")
print(f"\nTrain shape: {df_train_fe.shape}")
print(f"Val shape:   {df_val_fe.shape}")

Features after prep: 80

Train shape: (598647, 80)
Val shape:   (293095, 80)


In [6]:
y_train = df_train_fe["default"].values
y_val = df_val_fe["default"].values

x_train = df_train_fe.drop(columns=["default"])
x_val = df_val_fe.drop(columns=['default'])

x_train['term'] = x_train['term'].str.extract(r'(\d+)').astype(int)
x_val['term']   = x_val['term'].str.extract(r'(\d+)').astype(int)

categorical_cols = x_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = x_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumeric columns: {len(numeric_cols)} features")

Categorical columns (7): ['grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type']

Numeric columns: 72 features


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# numeric pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
    ]
)

# categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# both
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
    ]
) 

classifier = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='lbfgs',
        random_state=42
    ))
    ]
)

print("Pipeline built. Steps:")
for name, step in classifier.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

Pipeline built. Steps:
  preprocessor: ColumnTransformer
  classifier: LogisticRegression


In [8]:
import time

# Train
start = time.time()

classifier.fit(x_train, y_train)

train_time = time.time() - start

# Predictions
val_probs = classifier.predict_proba(x_val)[:, 1]

# metrics
auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)
brier = brier_score_loss(y_val, val_probs)

# Profit optimization
thresholds = np.linspace(0.05, 0.55, 100)

profits = [
    calculate_profit(df_val, val_probs < t)
    for t in thresholds
]

best_idx = int(np.argmax(profits))

best_threshold = float(thresholds[best_idx])
best_profit = float(profits[best_idx])

approval_rate = float((val_probs < best_threshold).mean())

results = log_run(results, {
    "run_name": "logreg_baseline",
    "model_type": "LogisticRegression",
    "best_threshold": best_threshold,
    "auc": float(auc),
    "pr_auc": float(pr_auc),
    "brier": float(brier),
    "profit_at_threshold": float(best_profit),
    "approval_rate": float(approval_rate),
    "train_time_seconds": float(train_time),
})

print("=== LOGISTIC REGRESSION ===")
print(f"Train time:     {train_time:.1f}s")
print(f"AUC:            {auc:.4f}")
print(f"PR-AUC:         {pr_auc:.4f}")
print(f"Brier:          {brier:.4f}")
print(f"Best threshold: {best_threshold:.4f}")
print(f"Approval rate:  {approval_rate:.2%}")
print(f"Profit:         ${best_profit:,.0f}")

go = next(r for r in results if r["run_name"] == "baseline_grade_only")

print("\n--- Comparison to baselines ---")
print(f"Grade-only AUC:    {go['auc']:.4f}  →  LogReg:  {auc:.4f}  (delta: {auc - go['auc']:+.4f})")
print(f"Grade-only profit: ${go['profit_at_threshold']/1e6:.1f}M  →  LogReg:  ${best_profit/1e6:.1f}M  (delta: ${(best_profit - go['profit_at_threshold'])/1e6:+.1f}M)")

=== LOGISTIC REGRESSION ===
Train time:     14.8s
AUC:            0.7157
PR-AUC:         0.4246
Brier:          0.2194
Best threshold: 0.3783
Approval rate:  35.83%
Profit:         $60,676,972

--- Comparison to baselines ---
Grade-only AUC:    0.6816  →  LogReg:  0.7157  (delta: +0.0340)
Grade-only profit: $42.8M  →  LogReg:  $60.7M  (delta: $+17.8M)


In [9]:
x_train_no_rate = x_train.drop(columns=["int_rate"])
x_val_no_rate = x_val.drop(columns=["int_rate"])

numeric_cols_no_rate = [
    c for c in numeric_cols
    if c != "int_rate"
]

preprocessor_no_rate = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols_no_rate),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

classifier_no_rate = Pipeline(steps=[
    ("preprocessor", preprocessor_no_rate),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=42
    ))
])

start = time.time()

classifier_no_rate.fit(x_train_no_rate, y_train)

train_time = time.time() - start

#Predictions
val_probs = classifier_no_rate.predict_proba(x_val_no_rate)[:, 1]

# Metrics
auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)
brier = brier_score_loss(y_val, val_probs)

# Profit optimization
thresholds = np.linspace(0.05, 0.55, 100)

profits = [
    calculate_profit(df_val, val_probs < t)
    for t in thresholds
]

best_idx = int(np.argmax(profits))

best_threshold = float(thresholds[best_idx])
best_profit = float(profits[best_idx])

approval_rate = float((val_probs < best_threshold).mean())

results = log_run(results, {
    "run_name": "logreg_no_int_rate",
    "model_type": "LogisticRegression",
    "excluded_feature": "int_rate",
    "best_threshold": best_threshold,
    "auc": float(auc),
    "pr_auc": float(pr_auc),
    "brier": float(brier),
    "profit_at_threshold": float(best_profit),
    "approval_rate": float(approval_rate),
    "train_time_seconds": float(train_time),
})

print("=== LOGISTIC REGRESSION (no int_rate) ===")
print(f"Train time:     {train_time:.1f}s")
print(f"AUC:            {auc:.4f}")
print(f"PR-AUC:         {pr_auc:.4f}")
print(f"Brier:          {brier:.4f}")
print(f"Best threshold: {best_threshold:.4f}")
print(f"Approval rate:  {approval_rate:.2%}")
print(f"Profit:         ${best_profit:,.0f}")

baseline = next(r for r in results if r["run_name"] == "logreg_baseline")
base_auc = baseline["auc"]
base_profit = baseline["profit_at_threshold"]

print("\n--- Comparison: with vs without int_rate ---")
print(f"With int_rate:    AUC {base_auc:.4f}, Profit ${base_profit/1e6:.1f}M")
print(f"Without int_rate: AUC {auc:.4f}, Profit ${best_profit/1e6:.1f}M")
print(f"Cost of dropping: AUC {auc - base_auc:+.4f}, Profit ${(best_profit - base_profit)/1e6:+.1f}M")

=== LOGISTIC REGRESSION (no int_rate) ===
Train time:     14.4s
AUC:            0.7146
PR-AUC:         0.4245
Brier:          0.2160
Best threshold: 0.3682
Approval rate:  34.45%
Profit:         $59,901,460

--- Comparison: with vs without int_rate ---
With int_rate:    AUC 0.7157, Profit $60.7M
Without int_rate: AUC 0.7146, Profit $59.9M
Cost of dropping: AUC -0.0011, Profit $-0.8M


In [10]:
pd.DataFrame(results).sort_values(
    "profit_at_threshold",
    ascending=False
)

,run_name,model_type,predicted_default_prob,auc,pr_auc,brier,profit_at_threshold,approval_rate,best_threshold,train_time_seconds,excluded_feature
2,logreg_baseline,LogisticRegression,NaN,0.715654,0.424605,0.219360,60676972.5,0.358256,0.378283,14.756142,NaN
3,logreg_no_int_rate,LogisticRegression,NaN,0.714598,0.424545,0.216012,59901460.0,0.344527,0.368182,14.401941,int_rate
1,baseline_grade_only,grade_lookup,NaN,0.681608,0.359212,0.166423,42848472.5,0.470684,0.125758,NaN,NaN
0,baseline_approve_all,constant,0.195381,0.500000,0.232832,0.180024,-206712975.0,1.000000,NaN,NaN,NaN


In [11]:
with open("../reports/results.json", "w") as f:
    json.dump(results, f, indent=2)